# 梯度下降法完整实现

## 目标

从零实现梯度下降法，理解其工作原理。

- 实现批量梯度下降
- 实现随机梯度下降 (SGD)
- 实现小批量梯度下降
- 对比不同变体的收敛特性

## 1. 数据准备

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

# 设置随机种子保证可复现
np.random.seed(42)

# 生成模拟数据
n_samples = 1000
n_features = 3

X = np.random.randn(n_samples, n_features)
true_w = np.array([3.5, -2.1, 1.5])
true_b = 4.0

y = np.dot(X, true_w) + true_b + np.random.randn(n_samples) * 0.5

print(f'数据形状: X={X.shape}, y={y.shape}')
print(f'真实参数: w={true_w}, b={true_b}')

## 2. 从零实现梯度下降

In [ ]:
class GradientDescent:
    """梯度下降法实现"""
    
    def __init__(self, learning_rate=0.01, n_iterations=1000, batch_size=None, random_state=None):
        self.learning_rate = learning_rate
        self.n_iterations = n_iterations
        self.batch_size = batch_size  # None = 批量, 1 = SGD, n = 小批量
        self.random_state = random_state
        self.weights = None
        self.bias = None
        self.loss_history = []
        
    def _initialize_parameters(self, n_features):
        """初始化参数"""
        if self.random_state:
            np.random.seed(self.random_state)
        self.weights = np.random.randn(n_features) * 0.01
        self.bias = 0.0
        
    def _predict(self, X):
        """预测"""
        return np.dot(X, self.weights) + self.bias
    
    def _compute_loss(self, y_true, y_pred):
        """计算均方误差"""
        return np.mean((y_true - y_pred) ** 2)
    
    def _compute_gradients(self, X, y, y_pred):
        """计算梯度"""
        n_samples = len(y)
        dw = -2 * np.dot(X.T, (y - y_pred)) / n_samples
        db = -2 * np.sum(y - y_pred) / n_samples
        return dw, db
    
    def fit(self, X, y):
        """训练模型"""
        n_samples, n_features = X.shape
        self._initialize_parameters(n_features)
        
        for iteration in range(self.n_iterations):
            if self.batch_size is None:
                # 批量梯度下降
                y_pred = self._predict(X)
                loss = self._compute_loss(y, y_pred)
                dw, db = self._compute_gradients(X, y, y_pred)
            elif self.batch_size == 1:
                # 随机梯度下降
                idx = np.random.randint(n_samples)
                X_batch, y_batch = X[idx:idx+1], y[idx:idx+1]
                y_pred = self._predict(X_batch)
                loss = self._compute_loss(y_batch, y_pred)
                dw, db = self._compute_gradients(X_batch, y_batch, y_pred)
            else:
                # 小批量梯度下降
                indices = np.random.choice(n_samples, self.batch_size, replace=False)
                X_batch, y_batch = X[indices], y[indices]
                y_pred = self._predict(X_batch)
                loss = self._compute_loss(y_batch, y_pred)
                dw, db = self._compute_gradients(X_batch, y_batch, y_pred)
            
            # 更新参数
            self.weights -= self.learning_rate * dw
            self.bias -= self.learning_rate * db
            
            self.loss_history.append(loss)
        
        return self
    
    def predict(self, X):
        """预测新数据"""
        return self._predict(X)

## 3. 训练模型 - 批量梯度下降

In [ ]:
# 批量梯度下降
gd_batch = GradientDescent(learning_rate=0.01, n_iterations=1000, batch_size=None, random_state=42)
gd_batch.fit(X, y)

print(f"批量梯度下降结果:")
print(f"权重: {gd_batch.weights}")
print(f"偏置: {gd_batch.bias:.4f}")
print(f"最终损失: {gd_batch.loss_history[-1]:.6f}")

## 4. 训练模型 - 随机梯度下降

In [ ]:
# 随机梯度下降 (需要更多迭代次数)
gd_sgd = GradientDescent(learning_rate=0.01, n_iterations=5000, batch_size=1, random_state=42)
gd_sgd.fit(X, y)

print(f"随机梯度下降结果:")
print(f"权重: {gd_sgd.weights}")
print(f"偏置: {gd_sgd.bias:.4f}")
print(f"最终损失: {gd_sgd.loss_history[-1]:.6f}")

## 5. 训练模型 - 小批量梯度下降

In [ ]:
# 小批量梯度下降
gd_minibatch = GradientDescent(learning_rate=0.01, n_iterations=1000, batch_size=32, random_state=42)
gd_minibatch.fit(X, y)

print(f"小批量梯度下降结果:")
print(f"权重: {gd_minibatch.weights}")
print(f"偏置: {gd_minibatch.bias:.4f}")
print(f"最终损失: {gd_minibatch.loss_history[-1]:.6f}")

## 6. 与 Scikit-learn 对比

In [ ]:
# Scikit-learn 的线性回归
sklearn_model = LinearRegression()
sklearn_model.fit(X, y)

y_pred_sklearn = sklearn_model.predict(X)
sklearn_mse = mean_squared_error(y, y_pred_sklearn)

print(f"Scikit-learn结果:")
print(f"权重: {sklearn_model.coef_}")
print(f"偏置: {sklearn_model.intercept_:.4f}")
print(f"MSE: {sklearn_mse:.6f}")

print("\n对比真实参数:")
print(f"真实权重: {true_w}")
print(f"真实偏置: {true_b}")

## 7. 收敛过程可视化

In [ ]:
plt.figure(figsize=(15, 5))

# 批量梯度下降
plt.subplot(1, 3, 1)
plt.plot(gd_batch.loss_history)
plt.xlabel('迭代次数')
plt.ylabel('损失')
plt.title('批量梯度下降')
plt.grid(True, alpha=0.3)

# 随机梯度下降 (只显示前1000次)
plt.subplot(1, 3, 2)
plt.plot(gd_sgd.loss_history[:1000], alpha=0.5)
plt.xlabel('迭代次数')
plt.ylabel('损失')
plt.title('随机梯度下降 (前1000次)')
plt.grid(True, alpha=0.3)

# 小批量梯度下降
plt.subplot(1, 3, 3)
plt.plot(gd_minibatch.loss_history)
plt.xlabel('迭代次数')
plt.ylabel('损失')
plt.title('小批量梯度下降 (batch_size=32)')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 8. 学习率的影响

In [ ]:
# 测试不同学习率
learning_rates = [0.001, 0.01, 0.05, 0.1, 0.5]

plt.figure(figsize=(15, 4))

for i, lr in enumerate(learning_rates):
    model = GradientDescent(learning_rate=lr, n_iterations=500, batch_size=None, random_state=42)
    model.fit(X, y)
    
    plt.subplot(1, len(learning_rates), i+1)
    plt.plot(model.loss_history)
    plt.xlabel('迭代次数')
    plt.ylabel('损失')
    plt.title(f'学习率 = {lr}')
    plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 9. 不同批量大小的影响

In [ ]:
# 测试不同批量大小
batch_sizes = [None, 1, 16, 64, 256]  # None=批量, 1=SGD
labels = ['批量', 'SGD', 'Mini-batch=16', 'Mini-batch=64', 'Mini-batch=256']

plt.figure(figsize=(12, 6))

for batch_size, label in zip(batch_sizes, labels):
    iterations = 1000 if batch_size != 1 else 5000
    model = GradientDescent(learning_rate=0.01, n_iterations=iterations, batch_size=batch_size, random_state=42)
    model.fit(X, y)
    
    # 为了对比，显示前1000次迭代
    loss_to_plot = model.loss_history[:1000]
    
    # SGD 使用移动平均平滑曲线
    if batch_size == 1:
        window = 50
        loss_to_plot = np.convolve(loss_to_plot, np.ones(window)/window, mode='same')
    
    plt.plot(loss_to_plot, label=label, alpha=0.7)

plt.xlabel('迭代次数')
plt.ylabel('损失')
plt.title('不同批量大小的收敛对比')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 10. 参数收敛过程

In [ ]:
# 增强版类，记录参数变化
class GradientDescentWithHistory(GradientDescent):
    """带参数历史记录的梯度下降"""
    
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.weight_history = []
        self.bias_history = []
    
    def fit(self, X, y):
        n_samples, n_features = X.shape
        self._initialize_parameters(n_features)
        
        for iteration in range(self.n_iterations):
            if self.batch_size is None:
                y_pred = self._predict(X)
                loss = self._compute_loss(y, y_pred)
                dw, db = self._compute_gradients(X, y, y_pred)
            elif self.batch_size == 1:
                idx = np.random.randint(n_samples)
                X_batch, y_batch = X[idx:idx+1], y[idx:idx+1]
                y_pred = self._predict(X_batch)
                loss = self._compute_loss(y_batch, y_pred)
                dw, db = self._compute_gradients(X_batch, y_batch, y_pred)
            else:
                indices = np.random.choice(n_samples, self.batch_size, replace=False)
                X_batch, y_batch = X[indices], y[indices]
                y_pred = self._predict(X_batch)
                loss = self._compute_loss(y_batch, y_pred)
                dw, db = self._compute_gradients(X_batch, y_batch, y_pred)
            
            self.weights -= self.learning_rate * dw
            self.bias -= self.learning_rate * db
            
            self.loss_history.append(loss)
            self.weight_history.append(self.weights.copy())
            self.bias_history.append(self.bias)
        
        return self

# 训练并记录参数
gd = GradientDescentWithHistory(learning_rate=0.01, n_iterations=200, batch_size=None, random_state=42)
gd.fit(X, y)

# 可视化参数收敛
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# 权重收敛
for i in range(n_features):
    weights_history = [w[i] for w in gd.weight_history]
    axes[0, 0].plot(weights_history, label=f'w{i+1}', alpha=0.7)
axes[0, 0].axhline(true_w[i], color='red', linestyle='--', label='真实值')
axes[0, 0].set_xlabel('迭代次数')
axes[0, 0].set_ylabel('权重值')
axes[0, 0].set_title('权重收敛过程')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# 偏置收敛
axes[0, 1].plot(gd.bias_history, label='偏置', alpha=0.7)
axes[0, 1].axhline(true_b, color='red', linestyle='--', label='真实值')
axes[0, 1].set_xlabel('迭代次数')
axes[0, 1].set_ylabel('偏置值')
axes[0, 1].set_title('偏置收敛过程')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# 损失收敛
axes[1, 0].plot(gd.loss_history, alpha=0.7)
axes[1, 0].set_xlabel('迭代次数')
axes[1, 0].set_ylabel('损失')
axes[1, 0].set_title('损失收敛过程')
axes[1, 0].set_yscale('log')
axes[1, 0].grid(True, alpha=0.3)

# 梯度大小
gradient_magnitudes = []
for i in range(len(gd.weight_history) - 1):
    gradient = np.linalg.norm(gd.weight_history[i+1] - gd.weight_history[i])
    gradient_magnitudes.append(gradient)
axes[1, 1].plot(gradient_magnitudes, alpha=0.7)
axes[1, 1].set_xlabel('迭代次数')
axes[1, 1].set_ylabel('梯度大小')
axes[1, 1].set_title('参数变化梯度大小')
axes[1, 1].set_yscale('log')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 总结

### 梯度下降的三种变体

| 方法 | 特点 | 优点 | 缺点 | 适用场景 |
|------|------|------|------|----------|
| 批量梯度下降 | 每次用全部数据 | 收敛稳定 | 大数据集慢 | 小数据集 |
| 随机梯度下降 (SGD) | 每次用1个样本 | 计算快，可在线学习 | 路径波动大 | 在线学习 |
| 小批量梯度下降 | 每次用小批数据 | 兼顾效率和稳定性 | 调参复杂 | 通用 |

### 关键要点

1. **学习率是关键参数**：
   - 太小：收敛慢
   - 太大：可能发散

2. **批量大小影响收敛特性**：
   - 批量：稳定但慢
   - SGD：快但波动
   - 小批量：最佳平衡

3. **收敛判断**：
   - 损失不再下降
   - 达到最大迭代次数
   - 梯度接近零

### 实践建议

- 从学习率 0.01 开始尝试
- 优先使用小批量梯度下降 (batch_size=32, 64, 128)
- 使用学习率衰减提高最终精度
- 监控损失曲线判断训练是否正常